In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch
from torch.optim import SGD, Adam
from transform import transform_log_scale, transform_cat
from sklearn.preprocessing import StandardScaler

In [32]:
df = df=pd.read_csv('Project_description_and_data/claims_train.csv')
len(df)

542410

In [19]:
#Preparing the data
df_scaled = transform_log_scale(df)
df_processed = transform_cat(df_scaled)
num_features = ['Exposure', 'VehPower', 'BonusMalus', 'VehAge_log', 'DrivAge_log', 'Density_log']
scaler = StandardScaler()
df_processed[num_features] = scaler.fit_transform(df_processed[num_features])
X = df_processed.drop(columns='ClaimNb')
y = df_processed['ClaimNb']

#splitting the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
class ModelDataset(Dataset):
    def __init__(self,x,y):
        self.x = x
        self.y = y
    def __getitem__(self,idx):
        return self.x[idx], self.y[idx]
    
    def __len__(self):
        return len(self.x)

In [21]:
X_train = X_train.astype("float32")
X_test = X_test.astype("float32")

In [33]:
X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy().reshape(-1, 1)

X_test_np = X_test.to_numpy()
y_test_np = y_test.to_numpy().reshape(-1, 1)

ds = ModelDataset(torch.from_numpy(X_train_np),torch.from_numpy(y_train_np))
ds_test = ModelDataset(torch.from_numpy(X_test_np),torch.from_numpy(y_test_np))

train_loader = DataLoader(ds, batch_size=516, shuffle=True)
test_loader = DataLoader(ds_test, batch_size=1, shuffle=True)

In [48]:
#This is our network

input_dim = X_train.shape[1]

class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 36)
        self.fc2 = nn.Linear(36, 24)
        self.fc3 = nn.Linear(24,12)
        self.fc4 = nn.Linear(12, 1)
    def forward(self, x):
        #x = x.view(-1, 64)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x
    
model = SimpleNN()

In [51]:
#Here we train the network
criterion = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=0.0005)

for epoch in range(50):
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = model(inputs.float())
        loss = criterion(outputs, labels.float())
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch + 1}, Loss: {running_loss / len(train_loader)}")

Epoch 1, Loss: 189.1728877864078
Epoch 2, Loss: 36.836199697933196
Epoch 3, Loss: 0.05692883787342257
Epoch 4, Loss: 0.05681729195783453
Epoch 5, Loss: 23.28357392633234
Epoch 6, Loss: 0.05717959213563364
Epoch 7, Loss: 0.057028641687278656
Epoch 8, Loss: 0.056900465037058
Epoch 9, Loss: 0.05681229322953825
Epoch 10, Loss: 0.05676832900788994
Epoch 11, Loss: 0.0567551723788808
Epoch 12, Loss: 0.05675499849065731
Epoch 13, Loss: 0.05674458321684464
Epoch 14, Loss: 0.05670756985977132
Epoch 15, Loss: 125.13014524714195
Epoch 16, Loss: 0.056976292935708335
Epoch 17, Loss: 0.05694100767923309
Epoch 18, Loss: 0.056895233300934225
Epoch 19, Loss: 0.05684576969367854
Epoch 20, Loss: 0.05680473758335204
Epoch 21, Loss: 0.05677381402388764
Epoch 22, Loss: 0.05676087921133152
Epoch 23, Loss: 0.05675626275274685
Epoch 24, Loss: 0.05675669874459966
Epoch 25, Loss: 0.05675500043524998
Epoch 26, Loss: 0.05675509826509408
Epoch 27, Loss: 0.056750852541552856
Epoch 28, Loss: 0.056751179255270505
Epoch

In [52]:
model.eval()
with torch.no_grad():
    outputs = model(torch.from_numpy(X_test_np).float())
    mse = nn.MSELoss()(outputs, torch.from_numpy(y_test_np).float())
    rmse = torch.sqrt(mse)
print("MSE:", mse.item())
print("RMSE:", rmse.item())

MSE: 0.05756144970655441
RMSE: 0.23991966247558594
